# Run Comparative Judging

This notebook demonstrates how to run the Comparative Judging system on remote viewing sessions fetched from the Social RV API.

## How Comparative Judging Works

1. **Fetch Session Data**: Get a session from the API along with its target and decoys
2. **Prepare Inputs**: Download and encode session files, target image, and decoy images
3. **Run the Judge**: The AI analyzes the session against all targets
4. **Get Results**: The system ranks each target from best match (1) to worst match

## Requirements

- OpenAI API key (for GPT-4 Vision)
- Social RV Research API key


In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import asyncio
from dotenv import load_dotenv
load_dotenv('../.env')

from comparative_judging import (
    SocialRVClient,
    SessionFile,
    TargetImage,
    judge_session_against_decoys,
    create_session_file_from_url,
    create_target_from_url,
)

# Initialize the API client
client = SocialRVClient()
print(f"✅ Client initialized")


## Find a Session with Decoys

First, let's find a session that has already been through comparative judging (has decoy IDs).


In [ ]:
# Fetch sessions and find one with decoys
print("📥 Fetching sessions to find one with decoys...")

sessions = client.fetch_all_sessions(
    include_low_value=False,
    max_sessions=100
)

# Find sessions with decoys
sessions_with_decoys = [s for s in sessions if s.decoy_ids and len(s.decoy_ids) >= 3]

print(f"\n✅ Found {len(sessions_with_decoys)} sessions with 3+ decoys")

if sessions_with_decoys:
    # Pick a session with session media
    sessions_with_media = [s for s in sessions_with_decoys if s.session_media_urls]
    sample_session = sessions_with_media[0] if sessions_with_media else sessions_with_decoys[0]
    
    print(f"\n📋 Selected session:")
    print(f"   ID: {sample_session.id}")
    print(f"   User: {sample_session.user_display_name}")
    print(f"   Target Coordinate: {sample_session.target_coordinate}")
    print(f"   Existing CJ Rank: {sample_session.comparative_judging_rank}")
    print(f"   Decoys: {len(sample_session.decoy_ids)}")
    print(f"   Session Media: {len(sample_session.session_media_urls)} files")
else:
    print("❌ No sessions with decoys found")


## Fetch Full Session Data

Now let's fetch the complete session data including the target and all decoys.


In [ ]:
# Fetch full session data with target and decoys
print("📥 Fetching full session data...")

full_data = client.get_session_with_decoys(sample_session.id)

session = full_data['session']
target = full_data['target']
decoys = full_data['decoys']

print(f"\n🎯 Correct Target:")
print(f"   ID: {target.id}")
print(f"   Description: {target.description[:100]}...")
print(f"   Image URL: {'Yes' if target.image_url else 'No'}")

print(f"\n🎭 Decoys ({len(decoys)}):")
for i, decoy in enumerate(decoys[:5]):
    print(f"   {i+1}. {decoy.id[:8]}... - {decoy.description[:50]}...")
if len(decoys) > 5:
    print(f"   ... and {len(decoys) - 5} more")

print(f"\n📁 Session Media ({len(session.session_media_urls)} files):")
for i, media in enumerate(session.session_media_urls[:3]):
    print(f"   {i+1}. {media.get('mime_type', 'unknown')}")


## Prepare Inputs for the Judge

Download and encode all the files needed for comparative judging.


In [ ]:
# Download and encode session files
print("📥 Downloading session files...")

session_files = []
for i, media in enumerate(session.session_media_urls):
    url = media.get('url')
    mime_type = media.get('mime_type', '')
    
    # Only include images (GPT-4V can process images directly)
    if url and mime_type.startswith('image/'):
        try:
            session_file = create_session_file_from_url(url, f"session_{i+1}.{mime_type.split('/')[-1]}")
            session_files.append(session_file)
            print(f"   ✅ Downloaded session file {i+1}: {mime_type}")
        except Exception as e:
            print(f"   ❌ Failed to download session file {i+1}: {e}")
    elif url and 'pdf' in mime_type:
        # Note: For PDFs, you would need to convert to images first
        print(f"   ⚠️  Skipping PDF file (would need conversion)")

print(f"\n📁 Prepared {len(session_files)} session files")

if not session_files:
    print("\n⚠️  WARNING: No image files were downloaded.")
    print("   The comparative judge requires at least one session image to analyze.")
    print("   This session only has PDF files. Consider:")
    print("   - Converting PDFs to images using PyMuPDF")
    print("   - Finding a different session with image files")


In [ ]:
# Download and encode target and decoy images
print("📥 Downloading target and decoy images...")

# Download correct target
print(f"\n🎯 Downloading correct target...")
if not target.image_url:
    raise ValueError("❌ Target has no image URL - cannot proceed with comparative judging")

target_image = create_target_from_url(
    target_id=target.id,
    description=target.description,
    image_url=target.image_url
)
print(f"   ✅ Downloaded target: {target.id[:8]}...")

# Download decoys (limit to 4 for demo to save API costs)
print(f"\n🎭 Downloading decoys...")
decoy_images = []
max_decoys = min(4, len(decoys))

for i, decoy in enumerate(decoys[:max_decoys]):
    if not decoy.image_url:
        print(f"   ⚠️  Skipping decoy {i+1}: no image URL")
        continue
    try:
        decoy_image = create_target_from_url(
            target_id=decoy.id,
            description=decoy.description,
            image_url=decoy.image_url
        )
        decoy_images.append(decoy_image)
        print(f"   ✅ Downloaded decoy {i+1}/{max_decoys}: {decoy.id[:8]}...")
    except Exception as e:
        print(f"   ❌ Failed to download decoy {i+1}: {e}")

print(f"\n✅ Prepared 1 target + {len(decoy_images)} decoys")


## Run the Comparative Judge

Now we'll run the AI judge to rank all targets against the session data.

⚠️ **Note**: This will make API calls to OpenAI (GPT-4 Vision), which has associated costs.


In [ ]:
# Validate inputs before running
if not session_files:
    raise ValueError("❌ Cannot run comparative judging: No session files available. See previous cell for options.")

if not decoy_images:
    raise ValueError("❌ Cannot run comparative judging: No decoy images were downloaded.")

# Run the comparative judge
print("🧠 Running Comparative Judge...")
print(f"   Session files: {len(session_files)}")
print(f"   Target + Decoys: 1 + {len(decoy_images)}")
print("\n   This may take 30-60 seconds...\n")

# Run the judge (async function)
result = await judge_session_against_decoys(
    session_files=session_files,
    target=target_image,
    decoys=decoy_images,
    model_override="gpt-4o"  # or "gpt-4o-mini" for faster/cheaper
)

print("✅ Judging complete!")


## Results


In [ ]:
# Display results
print("=" * 60)
print("📊 COMPARATIVE JUDGING RESULTS")
print("=" * 60)

print(f"\n🎯 Correct Target Rank: {result.correct_target_rank} out of {result.total_targets_ranked}")
print(f"✅ Verification Passed: {result.verification_passed}")
print(f"🔄 Attempts: {result.attempt_number}")

# Compare with existing rank
if session.comparative_judging_rank:
    print(f"\n📈 Comparison with existing CJ rank:")
    print(f"   Original: {session.comparative_judging_rank}")
    print(f"   New:      {result.correct_target_rank}")

print(f"\n📝 Overall Reasoning:")
print(f"   {result.overall_reasoning[:500]}...")

print(f"\n🏆 Rankings:")
for match in sorted(result.top_matches, key=lambda x: x.rank):
    is_correct = match.target_id == target.id
    marker = "🎯" if is_correct else "  "
    print(f"   {marker} Rank {match.rank}: {match.target_id[:12]}...")
    print(f"       {match.reasoning[:100]}...")


## Understanding the Results

### Key Metrics

- **Correct Target Rank**: Where the actual target ranked (1 = best match, higher = worse)
- **Verification Passed**: Whether the AI's reasoning was verified for consistency
- **Attempts**: Number of tries (retries happen if verification fails)

### Interpreting Rankings

- **Rank 1**: The AI thinks this target best matches the session data
- A session is considered "successful" if the correct target ranks 1st
- Random chance with 5 targets would give an average rank of 3

### Next Steps

- Run this on multiple sessions to analyze performance
- Compare results with the original CJ ranks from the platform
- Experiment with different numbers of decoys
- Try different models (gpt-4o vs gpt-4o-mini)
